# Human Development Index (HDI) Predictor Workflow

This Jupyter Notebook implements the end-to-end machine learning pipeline for predicting the Human Development Index (HDI) of countries. 
It follows the structured workflow of importing libraries, loading the raw dataset, performing exploratory data analysis (EDA), data preprocessing (handling null values and label encoding), model training using Linear Regression, and saving the serialized model.

### Step 1: Importing Required Libraries
We import standard packages for data manipulation (`pandas`, `numpy`), visualization (`matplotlib`, `seaborn`), machine learning (`sklearn`), and model serialization (`pickle`).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn import metrics
import os

print("Libraries successfully imported!")

### Step 2: Loading the Raw Dataset
We load the raw normalized dataset of 195 rows and 82 columns, which is prepared in the data pipeline.

In [ ]:
raw_data_path = "hdi_raw.csv"
if not os.path.exists(raw_data_path):
    print("Raw dataset not found! Running data download pipeline...")
    import download_data
    download_data.download_and_preprocess()

df = pd.read_csv(raw_data_path)
print("Dataset Shape:", df.shape)
df.head()

### Step 3: Exploratory Data Analysis (EDA)
We visualize relationships between the target variable (`HDI Score`) and development dimensions using the first 20 rows of the dataset to prevent clutter.

In [ ]:
data20 = df.head(20).copy()

# 1. Strip Plot of Mean Years of Schooling vs HDI Score
plt.figure(figsize=(10, 5))
sns.stripplot(x='Mean yrs of schooling', y='HDI Score', data=data20, size=8, color='cyan')
plt.title('Mean Years of Schooling vs HDI Score (First 20 Countries)')
plt.xlabel('Mean Years of Schooling')
plt.ylabel('HDI Score')
plt.show()

# 2. Scatter Plot of Life Expectancy vs HDI Score
plt.figure(figsize=(10, 5))
sns.scatterplot(x='Life expectancy', y='HDI Score', data=data20, color='tomato', s=100)
plt.title('Life Expectancy vs HDI Score (First 20 Countries)')
plt.xlabel('Life Expectancy (years)')
plt.ylabel('HDI Score')
plt.show()

# 3. Correlation Heatmap
numeric_cols = ['Life expectancy', 'Expected yrs of schooling', 'Mean yrs of schooling', 'GNI per capita', 'HDI Score']
plt.figure(figsize=(8, 6))
sns.heatmap(data20[numeric_cols].corr(), annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Correlation Matrix of Key Indicators')
plt.show()

### Step 4: Data Preprocessing and Feature Selection (Epic 4)
We select independent (features `X`) and dependent (target `y`) variables using their column indices in `df`:
- **Country Name**: Index 2
- **HDI Score**: Index 4 (Target `y`)
- **Life Expectancy**: Index 5
- **Expected yrs of schooling**: Index 6
- **Mean yrs of schooling**: Index 7
- **GNI per capita**: Index 8

In [ ]:
X_raw = df.iloc[:, [2, 5, 6, 7, 8]].copy()
y = df.iloc[:, 4]

print("Independent features columns:", X_raw.columns.tolist())
print("Dependent target variable:", y.name)

#### Checking and Handling Null Values
We use `isnull().sum()` to identify null inputs, and fill missing entries using the mean of each column.

In [ ]:
print("Null counts before preprocessing:")
print(X_raw.isnull().sum())

# Fill Null Values in X using column mean for numeric features
numeric_features = X_raw.columns[1:]
for col in numeric_features:
    mean_val = X_raw[col].mean()
    X_raw[col] = X_raw[col].fillna(mean_val)

print("\nNull counts after filling:")
print(X_raw.isnull().sum())

#### Categorical Variable Label Encoding
We encode the `Country Name` textual category into numeric labels suitable for Linear Regression parameters.

In [ ]:
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
X_raw.iloc[:, 0] = le.fit_transform(X_raw.iloc[:, 0].fillna('Unknown'))
X = X_raw
X.head()

### Step 5: Dividing the Dataset into Train and Test Sets (Epic 5)
We split the cleaned dataset into 80% training and 20% testing data.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Training set shape: {X_train.shape}")
print(f"Testing set shape: {X_test.shape}")

### Step 6: Fitting the Model and Generating Predictions (Epic 6)
We train the Linear Regression model, check its fit parameters, generate predictions, and evaluate test metrics.

In [ ]:
model = LinearRegression()
model.fit(X_train, y_train)
print("Model successfully trained!")

print("\nCoefficients:")
for col, coef in zip(X.columns, model.coef_):
    print(f"  {col}: {coef:.6f}")
print(f"  Intercept: {model.intercept_:.6f}")

y_pred = model.predict(X_test)
print("\nFirst 5 predictions:", y_pred[:5])
print("First 5 actual scores:\n", y_test.head(5).values)

#### Regression Evaluation Metrics
We calculate R-Squared ($R^2$), Mean Absolute Error (MAE), and Root Mean Squared Error (RMSE).

In [ ]:
r2 = metrics.r2_score(y_test, y_pred)
mae = metrics.mean_absolute_error(y_test, y_pred)
mse = metrics.mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)

print(f"Model Performance on Test Set:")
print(f"  R-Squared (R2): {r2:.4f}")
print(f"  MAE: {mae:.4f}")
print(f"  RMSE: {rmse:.4f}")

### Step 7: Saving the Trained Model (Epic 7)
We serialize the fitted Linear Regression model and the country LabelEncoder into `.pkl` files.

In [ ]:
with open("hdi_model.pkl", 'wb') as f:
    pickle.dump(model, f)
print("Model serialized to 'hdi_model.pkl'")

with open("country_encoder.pkl", 'wb') as f:
    pickle.dump(le, f)
print("Label Encoder serialized to 'country_encoder.pkl'")